<style>
.jp-RenderedMarkdown, .markdown-body { direction: rtl; text-align: right; font-family: Vazirmatn, Tahoma; }
.jp-RenderedMarkdown h1, h2 { color: #00f2ff !important; text-shadow: 0 0 8px #00f2ff88; }
.jp-RenderedMarkdown pre, .jp-RenderedMarkdown code {
    direction: ltr !important; text-align: left !important;
    display: inline-block; unicode-bidi: embed;
    background: #1e1e1e; color: #ce9178; border-radius: 6px; padding: 2px 6px;
}
.jp-RenderedMarkdown pre { display: block; padding: 14px; border: 1px solid #333; }
</style>


<div dir="rtl">

# 🚀 سناریوی آموزشی: انتقال به اعتبارسنجی مدرن با Pydantic

سلام دانشجویان عزیز! در این سناریو از بلاک‌های تو در تو‌ی `if/else` رها می‌شویم و کدی Production-Grade می‌نویسیم.

</div>


<div dir="rtl">

## صورت مسئله (سناریو)

سیستم ثبت محصول برای فروشگاه اینترنتی با این قوانین:

1. **شناسه محصول (`id`)**: عدد صحیح منجمد (Frozen).
2. **عنوان (`title`)**: بین ۳ تا ۵۰ کاراکتر.
3. **قیمت (`price`)**: بزرگتر از صفر و ضریبی از `0.01`.
4. **برچسب‌ها (`tags`)**: لیستی ۱ تا ۵ عضوی.
5. **زمان ایجاد (`created_at`)**: پیش‌فرض UTC، با محاسبه مجدد در هر نمونه.
6. **کلید امنیتی (`secret_api_key`)**: هرگز در خروجی Dict/JSON ظاهر نشود.
7. **کد کالا (`sku`)**: مطابق الگوی `PROD-[0-9]{4}`.

</div>


<div dir="rtl">

## گام اول: پیاده‌سازی با Vanilla Python

در پایتون استاندارد ناچاریم کلاسی سنگین با `__init__` شلوغ و مدیریت دستی استثنا بنویسیم:

</div>


In [ ]:
from datetime import datetime, timezone
import re


class Product:
    def __init__(
        self,
        id: int,
        title: str,
        price: float,
        tags: list[str],
        sku: str,
        secret_api_key: str,
        created_at: datetime = None,
    ):

        # ۱. اعتبارسنجی شناسه (ثابت بودن آن را بعداً در __setattr__ کنترل می‌کنیم)
        if not isinstance(id, int):
            raise TypeError("شناسه باید عدد صحیح باشد.")
        self._id = id

        # ۲. اعتبارسنجی عنوان
        if not isinstance(title, str):
            raise TypeError("عنوان باید رشته باشد.")
        if not (3 <= len(title) <= 50):
            raise ValueError("طول عنوان باید بین ۳ تا ۵۰ کاراکتر باشد.")
        self.title = title

        # ۳. اعتبارسنجی قیمت
        if not isinstance(price, (int, float)):
            raise TypeError("قیمت باید عدد باشد.")
        if price <= 0:
            raise ValueError("قیمت باید بزرگتر از صفر باشد.")
        if round(price % 0.01, 5) != 0:
            raise ValueError("قیمت باید ضریبی از 0.01 باشد.")
        self.price = float(price)

        # ۴. اعتبارسنجی برچسب‌ها (لیست)
        if not isinstance(tags, list):
            raise TypeError("برچسب‌ها باید یک لیست باشند.")
        if not (1 <= len(tags) <= 5):
            raise ValueError("تعداد برچسب‌ها باید بین ۱ تا ۵ عدد باشد.")
        for tag in tags:
            if not isinstance(tag, str):
                raise TypeError("هر برچسب باید رشته باشد.")
        self.tags = tags

        # ۵. اعتبارسنجی SKU با عبارات منظم (Regex)
        if not isinstance(sku, str):
            raise TypeError("کد SKU باید رشته باشد.")
        if not re.match(r"^PROD-[0-9]{4}$", sku):
            raise ValueError("فرمت SKU نامعتبر است. باید به صورت PROD-XXXX باشد.")
        self.sku = sku

        # ۶. کلید امنیتی
        self.secret_api_key = secret_api_key

        # ۷. مدیریت زمان ایجاد (حل مشکل Mutable Defaults به روش سنتی پایتون)
        if created_at is None:
            # اگر مستقیماً در تعریف آرگومان نوشته بودیم datetime.now()، در زمان کامپایل قفل می‌شد!
            self.created_at = datetime.now(timezone.utc)
        else:
            self.created_at = created_at

    # برای غیرقابل تغییر کردن فیلد شناسه (id) به صورت دستی
    def __setattr__(self, name, value):
        if name == "id" or name == "_id":
            if hasattr(self, "_id"):
                raise AttributeError(
                    "شناسه محصول منجمد (Frozen) است و قابل تغییر نیست."
                )
        super().__setattr__(name, value)

    @property
    def id(self):
        return self._id

    # سریال‌سازی دستی برای حذف کلید امنیتی از خروجی
    def to_dict(self):
        return {
            "id": self.id,
            "title": self.title,
            "price": self.price,
            "tags": self.tags,
            "sku": self.sku,
            "created_at": self.created_at.isoformat(),
            # فیلد secret_api_key عمداً حذف شده است
        }


# تست اجرای پایتون سنتی
try:
    product = Product(
        id=101,
        title="لپ‌تاپ گیمینگ ایسوس",
        price=1200.55,
        tags=["لپ‌تاپ", "ایسوس", "گیمینگ"],
        sku="PROD-5542",
        secret_api_key="super-secret-key-123",
    )
    print("محصول با موفقیت ساخته شد:", product.to_dict())

    # تست خطاها
    # product.id = 102 # خطا می‌دهد: AttributeError
    # invalid_product = Product(101, "A", -50, [], "INVALID-SKU", "secret") # کلی خطای دستی صادر می‌کند!
except Exception as e:
    print("خطا:", e)

<div dir="rtl" style="font-family: Vazirmatn, Tahoma, sans-serif; line-height: 1.8; color: #e0e0e0;">

<h2 style="color: #00f2ff; text-shadow: 0 0 8px rgba(0, 242, 255, 0.4); border-bottom: 2px solid #333; padding-bottom: 8px;">
گام دوم: بازنویسی هوشمندانه با Pydantic (گام‌به‌گام)
</h2>

<p>

</p>

<h3 style="color: #bc13fe; text-shadow: 0 0 6px rgba(188, 19, 254, 0.4); margin-top: 20px;">
۱. محدودیت‌های عددی (تلفیق با ویدیوی ۴۵)
</h3>
<p>
به جای اعتبارسنجی قیمت با شروط پیچیده، از کلاس <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">Field</code> پایدنتیک و مشخصه‌های <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">gt</code> (بزرگتر از) و <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">multiple_of</code> (ضریبی از) استفاده می‌کنیم:
</p>
<div dir="ltr" style="background: #1e1e1e; color: #50fa7b; padding: 10px 15px; border-radius: 6px; border: 1px solid #333; font-family: Consolas, monospace; margin: 10px 0; text-align: left;">
price: float = Field(gt=0, multiple_of=0.01) [2]
</div>

<h3 style="color: #bc13fe; text-shadow: 0 0 6px rgba(188, 19, 254, 0.4); margin-top: 20px;">
۲. محدودیت‌های رشته و دنباله (تلفیق با ویدیوی ۴۶)
</h3>
<ul>
    <li>برای محدود کردن طول عنوان، از <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">min_length=3</code> و <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">max_length=50</code> استفاده می‌کنیم [9].</li>
    <li>برای بررسی تعداد اعضای لیست برچسب‌ها (<code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">tags</code>)، پایدنتیک به ما اجازه می‌دهد همان محدودیت‌های <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">min_length</code> و <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">max_length</code> را روی دنباله‌ها (مانند لیست یا تاپل) اعمال کنیم [10].</li>
    <li>برای بررسی فرمت SKU، از مشخصه <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">pattern</code> با یک رشته خام (Raw String با پیشوند <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">r</code>) برای عبارات منظم استفاده می‌کنیم تا کاراکترهای بک‌اسلش اشتباه تفسیر نشوند [17, 18]:</li>
</ul>
<div dir="ltr" style="background: #1e1e1e; color: #50fa7b; padding: 10px 15px; border-radius: 6px; border: 1px solid #333; font-family: Consolas, monospace; margin: 10px 0; text-align: left;">
sku: str = Field(pattern=r"^PROD-[0-9]{4}$") [18]
</div>

<h3 style="color: #bc13fe; text-shadow: 0 0 6px rgba(188, 19, 254, 0.4); margin-top: 20px;">
۳. حل مشکل مقادیر پیش‌فرض متغیر و کار با Default Factory (تلفیق با ویدیوی ۴۷)
</h3>
<p>
در پایتون، اگر مقدار پیش‌فرض را مستقیماً برابر با یک شیء متغیر (Mutable) یا فراخوانی یک تابع بگذاریم (مثل <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">default=datetime.now()</code>)، با یک چالش جدی مواجه می‌شویم [20, 21]. این مقدار فقط <b>یک‌بار در زمان کامپایل</b> ارزیابی می‌شود و برای تمام دفعات بعدی و همه نمونه‌ها ثابت می‌ماند [21, 24].
</p>

<div dir="rtl" style="font-family: Vazirmatn, Tahoma, sans-serif; line-height: 1.8;">
<div style="border-right: 4px solid #00f2ff; background: rgba(0,242,255,0.05); padding: 15px; border-radius: 6px;">
<h4 style="color: #00f2ff;">💡 بخش ویژه: تفاوت کپی عمیق و کپی سطحی و هوشمندی پایدنتیک</h4>
<p>یکی از نقاط قوت پایدنتیک، مدیریت مقادیر پیش‌فرض متغیر (Mutable Defaults) مانند <code dir="ltr">list</code>، <code dir="ltr">set</code> و <code dir="ltr">dict</code> است. ابتدا دو مفهوم پایه را بشناسیم:</p>
<p><b>۱. کپی سطحی (Shallow Copy):</b><br>
پایتون فقط ظرف بیرونی شیء را کپی می‌کند، اما اشیاء داخلی همچنان به همان آدرس حافظه قبلی اشاره می‌کنند؛ تغییر در یکی، نمونه اصلی را نیز تغییر می‌دهد.</p>
<p><b>۲. کپی عمیق (Deep Copy):</b><br>
کپی کاملاً مستقل از شیء و تمام اشیاء داخلی آن در ناحیه جدید حافظه؛ هیچ اشتراکی بین نسخه جدید و اصلی وجود ندارد.</p>
<h4 style="color: #ffb86c;">چرا پایدنتیک هوشمند است؟</h4>
<p>در Data Classes اگر بنویسید <code dir="ltr">elements: list[int] = []</code> خطای <code dir="ltr">ValueError: mutable default ... is not allowed</code> می‌گیرید؛ اما پایدنتیک به‌طور خودکار برای هر نمونه جدید یک <b>Deep Copy</b> از مقدار پیش‌فرض تهیه می‌کند و از باگ‌های پنهان پروداکشن جلوگیری می‌کند.</p>
<h4 style="color: #ffb86c;">چه زمانی Default Factory لازم است؟</h4>
<p>برای مقادیری مثل زمان جاری یا UUID باید از <code dir="ltr">default_factory=lambda: datetime.now(timezone.utc)</code> استفاده کنیم، چون <code dir="ltr">default=datetime.now(timezone.utc)</code> فقط یک‌بار در زمان تعریف کلاس ارزیابی و برای همیشه ثابت می‌ماند.</p>
</div>
</div>


<h3 style="color: #bc13fe; text-shadow: 0 0 6px rgba(188, 19, 254, 0.4); margin-top: 20px;">
۴. پیکربندی‌های پیشرفته فیلدها (تلفیق با ویدیوی ۴۸)
</h3>
<ul>
    <li><b>تغییر از Lax به Strict</b>: به صورت پیش‌فرض پایدنتیک داده‌ها را تغییر نوع (Coerce) می‌دهد (مثلاً رشته <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">"100"</code> را به عدد <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">100</code> تبدیل می‌کند) [36]. ما می‌توانیم با <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">strict=True</code> در فیلد یا مدل، مانع این کار شویم تا داده‌ها با دقت و سخت‌گیرانه وارد شوند [37].</li>
    <li><b>فیلدهای منجمد (<code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">frozen=True</code>)</b>: برای فیلد <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">id</code> می‌خواهیم مطمئن شویم پس از ساخته شدن مقدارش قابل تغییر نیست. کافیست در تعریف فیلد، <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">frozen=True</code> قرار دهیم [45].</li>
    <li><b>مخفی کردن فیلد از سریال‌سازی (<code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">exclude=True</code>)</b>: برای فیلد <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">secret_api_key</code> به جای حذف دستی آن در خروجی، مقدار <code dir="ltr" style="background:#1e1e1e; color:#ff79c6; padding:2px 6px; border-radius:4px; font-family:Consolas, monospace;">exclude=True</code> را قرار می‌دهیم. در این صورت فیلد در شیء پایتونی قابل استفاده است اما در خروجی‌های دیکشنری یا JSON ظاهر نخواهد شد [47, 48].</li>
</ul>

</div>
